# LU-Zerlegung für Tridiagonalmatrizen

## Theorie und Ziele

Für *Tridiagonalmatrizen* lässt sich die *LU-Zerlegung* effizienter durchführen, als für vollbesetzte Matrizen. Solche Matrizen treten bei der numerischen Lösung von eindimensionalen Randwertproblemen (RWP) auf. Wir betrachten exemplarisch 

$$
     -u''(x) = f(x), \quad (0 < x < 1), \qquad u(0) = u_0, u(1) = u_n
$$

für vorgegebene Randwerte $u_0, u_n$. Durch **Diskretisierung** lässt sich daraus ein LGS mit tridiagonaler Koeffizentenmatrix $A$ für die Stützwerte $u_i = u(x_i)$ gewinnen. Wir teilen dazu das Intervall $[0,1]$ in $n$ Intervalle der Länge $h>0$ an den Teilungspunkten $x_0 = 0, x_1 = x_0 + h, \ldots, x_n = 1$. Daraus entsteht ein tridiagonales LGS $A u = b$ (s. Unterricht).

**Ziel** des Praktikums ist es, die Algorithmen für Tridiagonalmatrizen zu implementieren und an einem Beispiel zu testen. Obwohl $A$ für die hier betrachtete Problemklasse immer gleich aussieht ($2$ auf der Hauptdiagonalen, $-1$ auf den zwei Nebendiagonalen) soll die LU-Zerlegung allgemein für Tridiagonalmatrizen umgesetzt werden.

Da fast alle Einträge in $A$ verschwinden, wird die Matrix $A$ nicht als Matrix gespeichert, sondern nur die drei Diagonalen als Vektoren. *Eine* Möglichkeit besteht darin, eine $3 \times n$ Matrix zu verwenden (aber es gibt natürlich viele andere Varianten):

$$
        M = \begin{pmatrix} *      & a_{10} & \ldots & a_{n-2,n-3} & a_{n-1,n-2} \\
                            a_{00} & a_{11} & \ldots &  a_{n-2,n-2} & a_{n-1,n-1} \\
                            a_{01} & a_{12} & \ldots &  a_{n-2,n-1} & * \end{pmatrix}
$$

Die mit * bezeichneten Werte werden nicht verwendet. Das Resultat der LR-Zerlegung ist ebenfalls tridiagonal und kann auf dieselbe Art gespeichert werden.

## Aufgaben

### Aufgabe 1

Implementieren Sie die LR-Zerlegung effizient für Tridiagonalmatrizen (Algorithmus 6). Eine mögliche Vorgehensweise ist:

* Verwenden Sie den Thomas Algorithmus (Algorithmus 6) aus dem Skript
* Schreiben Sie eine kleine Hilfsfunktion ind(i, j), die Indices (i, j) aus der vollen Tridiagonalmatrix in Indices (i', j') in der kompakten Speicherung umrechnet.
* Ersetzen Sie dann im Thomas Algorithmus z.B. r[i, j] durch LR[*ind(i, j)] etc.

Die Funktion _ind_ ist sehr einfach und Sie können natürlich auch darauf verzichten und die Indices direkt umrechnen

In [1]:
import numpy as np

def LR_tri(A):
    n = A.shape[1]
    LR = np.zeros_like(A)
    # init:
    LR[0,0] = A[0,0]
    #
    for j in range(1, n):
        
        LR[j,j-1] = (A[j,j-1] / LR[j-1,j-1]) # l schreiben
        LR[j-1,j] = A[j-1,j] # r schreiben
        LR[j,j] = A[j,j] - (LR[j,j-1]* LR[j-1,j]) # r schreiben(diagonale)
    #     
    return LR

Testen Sie Ihre Umsetzung. Der folgende Testcode funktioniert, falls die Tridiagonalmatrix wie in der Einleitung beschrieben gespeichert wurde.

In [2]:
import numpy as np
n=5 # Grösse der Matrizen
# test LUT
for k in range(n):
#    A = np.random.rand(n, n)
# 1. Zufallswerte für die drei Diagonalen generieren
    haupt_diag = np.random.rand(n)       # n Elemente
    untere_neben = np.random.rand(n-1)   # n-1 Elemente
    obere_neben = np.random.rand(n-1)    # n-1 Elemente

# 2. Mit np.diag die volle Matrix zusammenbauen
# k=0 ist die Hauptdiagonale, k=-1 darunter, k=1 darüber
    A = np.diag(haupt_diag, k=0) + \
    np.diag(untere_neben, k=-1) + \
    np.diag(obere_neben, k=1)

    #print("Zufällige Tridiagonalmatrix A:\n", A)
    LR = LR_tri(A) ##a neu
   
    L = np.tril(LR,-1) + np.eye(n)
    R = np.triu(LR)

    print(np.linalg.norm(L@R-A))
    assert(np.linalg.norm(L@R-A) < 1e-10)

1.1102230246251565e-16
0.0
2.2887833992611187e-16
1.1102230246251565e-16
5.551115123125783e-17


### Aufgabe 2

Implementieren Sie die Vorwärts- und Rücksubstitution effizient für Tridiagonalmatrizen. Sie können fbSub für allgemeine Matrizen adaptieren. 

 * Die Matrix R besteht nur aus der Hauptdiagonalen und einer Nebendiagonalen und die Matrix L besteht nur aus einer Nebendiagonalen. Die "Summen" in den Zeilen 3 und 6 von Algorithmus 2 bestehen daher je aus einem einzigen Term. 
 * Achtung: die "Hauptdiagonale" von L, die konstant 1 ist, ist in der kompakten Darstellung nicht gespeichert!

In [3]:
"""
in: LU (output von LR_tri), Vektor b
out: Vektor x s.t. L@U@x == b
"""  
def fbSub_tri(LR, b):
    n = b.shape[0]
    
    y = np.zeros_like(b)
    y[0] = b[0] 
    << snipp >>
    return x

SyntaxError: invalid syntax (3702009978.py, line 10)

Testen Sie Ihre Umsetzung. Der folgende Testcode ist wiederum auf die oben beschriebene Speicherung der Matrizen ausgelegt.

In [ ]:
# test fbSubsT
for k in range(10):
    m = np.random.rand(3,n) 
    m[0][0], m[-1][-1] = 0, 0
    A = np.diag(m[0][1:], k=-1) + np.diag(m[1], k=0) + np.diag(m[2][:-1], k=1) #testhalber volle Matrix erzeugen
    
    x1 = np.random.rand(n)   # Lösungsvektor
    b = A@x1                   # rechte Seite des LGS
    
    LR = LR_tri(m)
    x2 = fbSub_tri(LR, b)
    assert(np.linalg.norm(x1-x2) < 1e-10)

### Aufgabe 3

Wenden Sie die oben implementierten Algorithmen auf das in der Einleitung genannte RWP an, plotten Sie die numerische Lösung zusammen mit der exakten Lösung. Die tridiagonale Matrix des LGS ist nun gegeben durch die finite Differenzen Diskretisierung (Beispiel 14 im Skript).

In [ ]:
N = 100
x = np.linspace(0,1,N+1)

In [ ]:
# Dirichlet Randwerte
y0 = 0
yN = 0

In [ ]:
# System Matrix
A = np.zeros( (3, N - 1) ) # Das LGS muss für die inneren Punkte x1 .. x[N-1] gelöst werden 
A[0,1:]  = -np.ones(N-2)
A[1]     = 2*np.ones(N-1)
A[2,:-1] = -np.ones(N-2)

Im Beispiel benutzen wir $f(x) = 1$.

In [ ]:
# Rechte Seite
h = 1./N
b = - h**2 * np.ones(N-1)
b[0]  += y0
b[-1] += yN

Lösung berechnen und visualisieren:

In [ ]:
import matplotlib.pyplot as plt

LR = LR_tri(A)
y = np.zeros((N+1))
y[0] = y0;   # Randwert links
y[-1] = yN;  # Randwert rechts
y[1:-1] = fbSub_tri(LR, b)

ye = 0.5*x*(x-1); # Loesung von y''(x) = 1, y(0) = y(1) = 0

plt.plot(x, y)
plt.plot(x, ye,'--')
plt.grid()
plt.show()

NameError: name 'LR_tri' is not defined

### Abgabe

- Aufgabe 1: Funktion zur Berechnung der LU-Zerlegung für tridiagonal Matrizen
- Aufgabe 2: Funktion zum Lösen des Gleichungssystem mittels Vorwärts- / Rückwärtseinsetzen
- Aufgabe 3: Anwendung auf ein eindimensionales Randwertproblem

Kurzer Bericht mit den Ergebnissen und python Code.